In [57]:
from pathlib import Path
from rapidocr import RapidOCR
from rapidocr.utils.output import RapidOCROutput
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json 
from pydantic import BaseModel, Field
from ollama import chat


In [58]:
from dataclasses import dataclass


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        if (folder / ".git").exists():
            return folder
    raise FileNotFoundError(f"No git repo found above {start}")


REPO_ROOT = find_repo_root()

train_path = REPO_ROOT / "images" / "train"

@dataclass
class Sample:
    path: Path     
    img: np.ndarray  # BGR color


def load_images(folder: Path, limit: int | None = None) -> list[Sample]:
    samples = []

    for file_path in sorted(Path(folder).resolve().iterdir()):
        if not file_path.is_file():
            continue

        img = cv2.imread(str(file_path))
        if img is None or img.size == 0:
            continue

        try:
            rel_path = file_path.relative_to(REPO_ROOT)
        except ValueError:  # image lives outside the repo: keep the absolute path
            rel_path = file_path
        samples.append(Sample(rel_path, img))

        if limit is not None and len(samples) >= limit:
            break

    return samples

samples = load_images(folder=train_path, limit=5)

In [59]:
engine = RapidOCR()
def run_ocr(img: np.ndarray) -> RapidOCROutput | None:
    result = engine(img) 
    if not isinstance(result, RapidOCROutput) or result.boxes is None:
        return None
    return result


def get_ocr_res(result, img_shape, normalize=True):
    #Using rapidOCR

    h, w = img_shape[:2]
    items = []

    for box, txt, score in zip(result.boxes, result.txts, result.scores):

        #get lower left and upper right corners of bounding box from rapidocr result
        x1, y1 = box.min(axis=0)
        x2, y2 = box.max(axis=0)

        if normalize: 
            #ratio of coordinate to axis dim out of 1000
            x1, x2 = x1 / w * 1000, x2 / w * 1000 
            y1, y2 = y1 / h * 1000, y2 / h * 1000   # y is scaled by image height, not width

        items.append({
            "text": txt, 
            "box": [int(x1), int(y1), int(x2), int(y2)],
            "score": round(float(score), 2)
        })

    items.sort(key=lambda i: (i["box"][1], i["box"][0]))   # top-to-bottom, then left-to-right
    return items

In [ ]:
from typing import Literal

# Blueprint: labels/schema.json, trimmed for speed: no raw_text, nested objects flattened,
# and sender / merchant / confidence / recipient_* dropped. Fewer output tokens = faster generation.
# `image_file` is added from the file name in code, so the model does not output it.
# recipient_account and recipient_name were removed because the model looped on them (it repeated digits /
# "300.00" until num_predict). max_length did NOT stop that, so it is not relied on here.

class Receipt(BaseModel):
    provider: Literal["GCash", "Maya", "BDO", "BPI", "MariBank", "GoTyme", "Wise", "other"] | None = None
    transaction_type: Literal["send money", "cash in", "cash out", "bill payment",
                              "QR payment", "remittance", "other"] | None = None
    # required by labels/schema.json
    amount: float | None
    currency: str = "PHP"
    fee: float | None
    total: float | None
    date: str | None               # ISO 8601, YYYY-MM-DD
    time: str | None = None        # HH:MM, 24-hour
    reference_number: str | None   # alphanumeric, no spaces or hyphens


# Ollama's `format=` only constrains the JSON shape. Field descriptions are NOT shown to the model,
# so what each field means has to be spelled out in the prompt.
INTRO = """You extract data from screenshots of Philippine bank / e-wallet transfer receipts.
Return one JSON object that follows the provided schema."""

FIELD_RULES = """Rules:
- Use only information that is actually present. Do not guess. Use null when a value is not visible.

Fields:
- provider: the bank or e-wallet the money was sent FROM ("Sent via ...", "From ..."). Not the recipient's bank. Use "other" if it is not in the allowed list.
- transaction_type: send money, cash in, cash out, bill payment, QR payment, remittance, or other.
- amount: the amount transferred, as a number with no symbol or commas. Not the fee and not the total including fee.
- currency: defaults to "PHP".
- fee: the service charge as a number. 0 if shown as free. null if not shown. Never guess the fee.
- total: amount plus fee, only if printed.
- date: YYYY-MM-DD only (for example "May 20, 2026" becomes "2026-05-20"). time: HH:MM in 24-hour format, only if shown.
- reference_number: the reference / confirmation / trace number with no spaces or hyphens.

Output JSON only, no explanations."""

# image only
SYSTEM_PROMPT = INTRO + "\n\n" + FIELD_RULES

# OCR text (+ optional image)
HYBRID_PROMPT = INTRO + """

INPUT
You receive OCR output as a JSON list of {text, box, score}, in reading order.
box is [x1, y1, x2, y2], normalized 0-1000 from the top-left. score is OCR confidence.
The receipt image may also be attached.

OCR RULES
1. Every value you return must be copied from an OCR "text" field. Do not retype digits from memory or from the image.
2. Use box positions and nearby label text (and the image, if attached) only to decide which text is the amount,
   date, provider, etc. Never use them to change a value.
3. If no OCR text supports a field, return null.

""" + FIELD_RULES

In [63]:
import time

hybrid_rows = []
failed_rows = []   # images where the model reply was cut off or invalid; the loop keeps going
timing_rows = []

STRUCT_MODEL = "qwen3-vl:2b-instruct"
for s in samples:
    t0 = time.perf_counter()
    result = run_ocr(s.img)
    items = get_ocr_res(result, s.img.shape) if result else []
    ocr_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    r = chat(
        model=STRUCT_MODEL,
        messages=[
            {"role": "system", "content": HYBRID_PROMPT},
            {"role": "user","content": "OCR output:\n" + json.dumps(items, ensure_ascii=False, separators=(",", ":")),
            # "images": [str(REPO_ROOT / s.path)]
             },
        ],
        format=Receipt.model_json_schema(),
        keep_alive="30m",   # keep the model loaded between calls
        options={"temperature": 0, "num_ctx": 8192, "num_predict": 512},   # num_predict caps a runaway reply
    )
    chat_s = time.perf_counter() - t0

    if not r.message.content.strip() or r.done_reason == "length":   # cut off: record why and move on
        failed_rows.append({
            "image_file": s.path.name, "done_reason": r.done_reason, "output_tokens": r.eval_count,
            "start": r.message.content[:150], "end": r.message.content[-150:],   # the end shows which field looped
        })
        continue
    receipt = Receipt.model_validate_json(r.message.content)
    hybrid_rows.append({"image_file": s.path.name, **receipt.model_dump()})

    # Ollama reports durations in nanoseconds
    timing_rows.append({
        "image_file": s.path.name,
        "ocr_s": round(ocr_s, 2),
        "chat_s": round(chat_s, 2),
        "load_s": round((r.load_duration or 0) / 1e9, 2),
        "prompt_tokens": r.prompt_eval_count,
        "prompt_s": round((r.prompt_eval_duration or 0) / 1e9, 2),
        "output_tokens": r.eval_count,
        "output_s": round((r.eval_duration or 0) / 1e9, 2),
        "output_tok_per_s": round((r.eval_count or 0) / ((r.eval_duration or 1) / 1e9), 1),
    })

if failed_rows:
    print(f"{len(failed_rows)} of {len(samples)} images failed:")
    display(pd.DataFrame(failed_rows))
display(pd.DataFrame(timing_rows))
pd.json_normalize(hybrid_rows)  # flattens nested fields, e.g. amount.value

4 of 5 images failed:


,image_file,done_reason,output_tokens,start,end
0,002.jpg,length,512,"{\n ""amount"": 300,\n ""fee"": 30,\n ""total"": ...",0.00 300.00 300.00 300.00 300.00 300.00 300.00...
1,003.jpg,length,512,"{\n ""amount"": 300,\n ""fee"": 0,\n ""total"": 3...",:03 PM 31 Aug 2026 at 12:03 PM 31 Aug 2026 at ...
2,004.jpg,length,512,"{\n ""amount"": 600,\n ""fee"": 0,\n ""total"": 6...",the recipient name based on the receipt conte...
3,005.jpg,length,512,"{\n ""amount"": 300,\n ""fee"": 0,\n ""total"": 3...",o****a C.) (Jo****a C.) (Jo****a C.) (Jo***...


,image_file,ocr_s,chat_s,load_s,prompt_tokens,prompt_s,output_tokens,output_s,output_tok_per_s
0,001.jpg,0.99,3.18,0.0,2162,0.03,114,2.96,38.5


,image_file,provider,status,transaction_type,amount,currency,fee,total,date,time,reference_number,recipient_name
0,001.jpg,None,None,send money,300.0,PHP,10.0,310.0,2026-05-23,None,1779521526959,ne rider sible ways u vays fas side of the way...


In [62]:
import re

# Keyword rules for the sender's provider, run on the RapidOCR lines (no model needed).
PROVIDER_PATTERNS = {
    "GCash":    r"g[\s-]?cash",
    "Maya":     r"\bmaya\b|paymaya",
    "BDO":      r"\bbdo\b|banco de oro",
    "BPI":      r"\bbpi\b|bank of the philippine islands",
    "MariBank": r"mari[\s-]?bank",
    "GoTyme":   r"go[\s-]?tyme",
    "Wise":     r"\bwise\b",
}
PROVIDER_RX = {name: re.compile(p, re.I) for name, p in PROVIDER_PATTERNS.items()}

# Labels that introduce the SENDER's provider. "To" / "Transfer to" are deliberately not here:
# the recipient's provider (e.g. "G-Xchange, Inc (GCash)") must not be picked up.
SENDER_ANCHOR = re.compile(r"^\s*(sent\s*via|from)\b", re.I)


def providers_in(text: str) -> list[str]:
    return [name for name, rx in PROVIDER_RX.items() if rx.search(text)]


def detect_provider(items: list[dict], window: int = 4) -> tuple[str | None, str | None]:
    """Return (provider, how). `items` come from get_ocr_res (already in reading order)."""
    texts = [i["text"] for i in items]

    # 1) anchored: a provider on, or within `window` lines after, a "Sent via" / "From" line
    for idx, t in enumerate(texts):
        if SENDER_ANCHOR.search(t):
            for u in texts[idx: idx + 1 + window]:
                found = providers_in(u)
                if found:
                    return found[0], "anchor"

    # 2) fallback: exactly one provider is mentioned anywhere on the receipt
    mentioned = {p for t in texts for p in providers_in(t)}
    if len(mentioned) == 1:
        return mentioned.pop(), "unique"

    return None, None   # let the model decide


# Compare the rules with the model's provider on the samples
model_provider = {r["image_file"]: r["provider"] for r in hybrid_rows} if "hybrid_rows" in globals() else {}
rule_rows = []
for s in samples:
    result = run_ocr(s.img)
    items = get_ocr_res(result, s.img.shape) if result else []
    rule, how = detect_provider(items)
    model = model_provider.get(s.path.name)
    rule_rows.append({
        "image_file": s.path.name,
        "rule_provider": rule,
        "rule_match": how,
        "model_provider": model,
        "final_provider": rule or model,   # rules win, the model is the fallback
    })

pd.DataFrame(rule_rows)

,image_file,rule_provider,rule_match,model_provider,final_provider
0,001.jpg,BPI,anchor,None,BPI
1,002.jpg,GCash,anchor,None,GCash
2,003.jpg,GoTyme,anchor,None,GoTyme
3,004.jpg,GCash,anchor,None,GCash
4,005.jpg,MariBank,anchor,None,MariBank


In [ ]:
# Keyword rules for the reference number, run on the RapidOCR lines (no model needed).
# Labels matched: "ref no." / "ref. no.", "reference number" / "reference no.", "ref id" / "reference id"
REF_LABEL = re.compile(r"\bref(?:erence)?\.?\s*(?:no\.?|number|id)(?![a-z])", re.I)

# A line that starts with another field label is never a reference value
OTHER_LABEL = re.compile(r"^\s*(date|amount|fee|total|from|to|sent|transfer|trace|status|time)\b", re.I)


def _digits(s: str) -> int:
    return len(re.findall(r"\d", s))


def _clean_ref(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]", "", s).upper()   # alphanumeric only: no spaces, hyphens or dots


def detect_reference(items: list[dict], window: int = 2) -> tuple[str | None, str | None]:
    """Return (reference_number, how). `items` come from get_ocr_res (reading order, 0-1000 boxes)."""
    for idx, it in enumerate(items):
        m = REF_LABEL.search(it["text"])
        if not m:
            continue

        # 1) value on the same line as the label: "Ref No. 8041 012"
        rest = it["text"][m.end():].strip(" :#.-")
        if _digits(rest) >= 4:
            value = rest
            # the number may wrap onto the next row: "Ref No. 8041 012" / "628744"
            if idx + 1 < len(items):
                nxt = items[idx + 1]
                if re.fullmatch(r"[\d\s-]{4,}", nxt["text"].strip()) and 0 <= nxt["box"][1] - it["box"][3] <= 25:
                    value += nxt["text"]
            return _clean_ref(value), "same-line"

        # 2) value on one of the next lines: "Reference No." / "ITO260831040302011"
        for nxt in items[idx + 1: idx + 1 + window]:
            t = nxt["text"].strip()
            if OTHER_LABEL.search(t):
                break
            if _digits(t) >= 4:
                return _clean_ref(t), "next-line"

    return None, None   # let the model decide


# Compare the rules with the model's reference_number on the samples
model_ref = {r["image_file"]: r["reference_number"] for r in hybrid_rows} if "hybrid_rows" in globals() else {}
ref_rows = []
for s in samples:
    result = run_ocr(s.img)
    items = get_ocr_res(result, s.img.shape) if result else []
    rule, how = detect_reference(items)
    model = model_ref.get(s.path.name)
    ref_rows.append({
        "image_file": s.path.name,
        "rule_ref": rule,
        "rule_match": how,
        "model_ref": model,
        "final_ref": rule or model,   # rules win, the model is the fallback
    })

pd.DataFrame(ref_rows)

,image_file,rule_ref,rule_match,model_ref,final_ref
0,001.jpg,464438,same-line,1779521526959,464438


- problems with too much context long inference times -> switched to instruct from thinking
- problems with looping -> shorter schema, 